# 02 Uncertainty walkthrough
Dirichlet quantities, the three losses, and calibration, with numbers you can check by hand.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else '.')
import numpy as np, torch
from src.models.evidential import dirichlet_summary
from src.training.losses import soft_dice_loss, evidential_ce, evidential_kl
# 3 voxels, K=2 (bg, tumour); the worked example of docs/learn/06
ev = torch.tensor([[0.2, 9.0], [0.4, 0.6], [12.0, 0.1]]).T.reshape(1, 2, 3)
y = torch.tensor([[1, 1, 0]])
print('evidence', tuple(ev.shape), 'labels', tuple(y.shape))

evidence (1, 2, 3) labels (1, 3)


In [2]:
d = dirichlet_summary(ev)
for k in ['alpha', 'S', 'p', 'u']: print(k, tuple(d[k].shape), d[k].squeeze().numpy().round(4).tolist())

alpha (1, 2, 3) [[1.2000000476837158, 1.399999976158142, 13.0], [10.0, 1.600000023841858, 1.100000023841858]]
S (1, 1, 3) [11.199999809265137, 3.0, 14.100000381469727]
p (1, 2, 3) [[0.1071000024676323, 0.4666999876499176, 0.921999990940094], [0.8928999900817871, 0.53329998254776, 0.07800000160932541]]
u (1, 1, 3) [0.1785999983549118, 0.666700005531311, 0.14180000126361847]


In [3]:
print('Dice loss        ', round(soft_dice_loss(d['p'], y).item(), 4), '(hand: 0.186)')
print('evidential CE   ', round(evidential_ce(ev, y).item(), 4), '(hand: 0.333)')
print('KL to uniform   ', round(evidential_kl(ev, y).item(), 4), '(hand: 0.024)')

Dice loss         0.186 (hand: 0.186)
evidential CE    0.3333 (hand: 0.333)
KL to uniform    0.0236 (hand: 0.024)


## No evidence vs lots of evidence

In [4]:
for e in [(0, 0), (3, 0), (30, 0), (30, 30)]:
    s = dirichlet_summary(torch.tensor(e, dtype=torch.float32).reshape(1, 2, 1, 1, 1))
    print('evidence', e, '-> p', s['p'].flatten().numpy().round(3).tolist(), 'u', round(s['u'].item(), 3))

evidence (0, 0) -> p [0.5, 0.5] u 1.0
evidence (3, 0) -> p [0.800000011920929, 0.20000000298023224] u 0.4
evidence (30, 0) -> p [0.968999981880188, 0.03099999949336052] u 0.062
evidence (30, 30) -> p [0.5, 0.5] u 0.032


## Calibration
A model that is always 90% confident but right 70% of the time has ECE about 0.2.

In [5]:
from src.metrics.calibration import ece, reliability_bins
rng = np.random.default_rng(0); n = 50_000
prob = np.stack([np.full(n, 0.1), np.full(n, 0.9)])
labels = (rng.random(n) < 0.7).astype(int)  # argmax is class 1; it is right when label == 1
print('ECE', round(ece(prob, labels), 3))
b = reliability_bins(prob, labels)
print('non-empty bins', int((b['count'] > 0).sum()), 'of', len(b['count']))

ECE 0.201
non-empty bins 1 of 15


## Uncertainty should point at errors

In [6]:
from src.metrics.calibration import uncertainty_error_overlap
label = (rng.random((20, 20, 20)) < 0.3).astype(int); pred = label.copy()
flip = rng.random(label.shape) < 0.1; pred[flip] = 1 - pred[flip]
good_u = flip * 0.9 + 0.05 * rng.random(label.shape); random_u = rng.random(label.shape)
print('informative u:', {k: round(v, 3) for k, v in uncertainty_error_overlap(good_u, pred, label).items() if k in ('auroc', 'overlap_dice')})
print('random u     :', {k: round(v, 3) for k, v in uncertainty_error_overlap(random_u, pred, label).items() if k in ('auroc', 'overlap_dice')})

informative u: {'overlap_dice': 1.0, 'auroc': 1.0}
random u     : {'overlap_dice': 0.109, 'auroc': 0.488}
